# ITO5202 Assessment 1 — Analysing Historical Data with System Performance

**Student ID:** 29701201  **Unit:** ITO5202
**Dataset:** Brazilian E-Commerce Public Dataset by Olist ([Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce))

## Environment Setup 

Spark runs in local mode inside a Docker container, with the driver acting as the single executor. The configuration choices below are made deliberately because they affect every later measurement:

- `local[*]` uses every core the container exposes, so `defaultParallelism` equals the container's core count.
- `spark.driver.memory` must be set before the JVM starts, so this cell should be the first Spark call after a kernel restart.
- `spark.sql.session.timeZone = UTC`. The Olist timestamps are Brazilian local times with no zone attached. Parsing them in UTC stops the container's time zone (and any daylight-saving gaps) from silently shifting or nulling timestamps, which would otherwise change quarter boundaries and delivery delays.
- `spark.sql.shuffle.partitions` is reduced from the default 200 to 2 × cores. The full pipeline processes roughly 110k line items. 200 shuffle partitions would create mostly tiny tasks whose scheduling overhead exceeds their work. 

In [1]:
# ---------------------------------------------------------------
# Imports
# ---------------------------------------------------------------
import os          # file paths and CPU count
import platform    # Python / OS version for the environment table

import pandas as pd  

# Spark configuration and entry points
from pyspark import SparkConf
from pyspark.sql import SparkSession

# DataFrame functions, window specifications and schema types
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

In [2]:
# ---------------------------------------------------------------
# Spark configuration
# ---------------------------------------------------------------

master = "local[*]"


app_name = "ITO5202_A1_Olist_29701201"

# Set up configuration parameters for Spark
spark_conf = (
    SparkConf()
    .setMaster(master)
    .setAppName(app_name)
    # Driver memory: in local mode the driver is also the executor, so this
    # is all the memory Spark has. It only takes effect when the JVM starts,
    # so restart the kernel before running this cell.
    .set("spark.driver.memory", "4g")
    # Parse timestamps exactly as written in the CSVs (no time-zone shifting)
    .set("spark.sql.session.timeZone", "UTC")
    # Hide console progress bars so the PDF export stays clean
    .set("spark.ui.showConsoleProgress", "false")
)

# ---------------------------------------------------------------
# SparkSession 
# ---------------------------------------------------------------
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")  # show errors only, to keep outputs readable

# ---------------------------------------------------------------
# Shuffle partitions
# ---------------------------------------------------------------
# After a shuffle (groupBy, join, window), data is split into this many
# partitions. 2 x cores gives each core about two tasks: enough to keep all
# cores busy without the overhead of the default 200 near-empty tasks.
SHUFFLE_PARTITIONS = sc.defaultParallelism * 2
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)

print(f"Spark {spark.version} session started: {app_name}")
print(f"Shuffle partitions set to {SHUFFLE_PARTITIONS} "
      f"({sc.defaultParallelism} cores x 2)")

Spark 4.1.1 session started: ITO5202_A1_Olist_29701201
Shuffle partitions set to 16 (8 cores x 2)


In [3]:
# ---------------------------------------------------------------
# Execution environment summary 
# ---------------------------------------------------------------
def container_memory_limit():
    """Return the memory limit Docker has placed on this container, if any.
    Checks cgroup v2 first, then cgroup v1."""
    for path in ("/sys/fs/cgroup/memory.max",
                 "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            raw = open(path).read().strip()
            if raw != "max" and int(raw) < 1 << 60:   # very large value = no limit
                return f"{int(raw) / 1024**3:.1f} GB"
            return "no limit set"
        except (OSError, ValueError):
            continue
    return "unknown"


env = {
    "Execution mode": f"local ({sc.master}), Docker container",
    "Spark version": spark.version,
    "Python version": platform.python_version(),
    "OS (container)": f"{platform.system()} {platform.release()}",
    "CPU cores visible to container": os.cpu_count(),
    "defaultParallelism": sc.defaultParallelism,
    "Driver memory": spark.conf.get("spark.driver.memory", "default (1g)"),
    "Container memory limit": container_memory_limit(),
    "spark.sql.shuffle.partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    # AQE can re-optimise plans at runtime (e.g. coalescing partitions,
    # switching join strategy). It appears as AdaptiveSparkPlan in explain().
    "spark.sql.adaptive.enabled (AQE)": spark.conf.get("spark.sql.adaptive.enabled"),
    # Tables smaller than this are broadcast to every task instead of shuffled
    "spark.sql.autoBroadcastJoinThreshold": spark.conf.get("spark.sql.autoBroadcastJoinThreshold"),
    "Session time zone": spark.conf.get("spark.sql.session.timeZone"),
    "Spark Web UI": "http://localhost:4040",
}

pd.DataFrame(env.items(), columns=["Setting", "Value"])

,Setting,Value
0,Execution mode,"local (local[*]), Docker container"
1,Spark version,4.1.1
2,Python version,3.13.12
3,OS (container),Linux 7.0.12-linuxkit
4,CPU cores visible to container,8
5,defaultParallelism,8
6,Driver memory,4g
7,Container memory limit,no limit set
8,spark.sql.shuffle.partitions,16
9,spark.sql.adaptive.enabled (AQE),true


## Data Loading


Every table is loaded with a hand-written `StructType` schema rather than `inferSchema=True`, for three reasons:

1. Correctness: Inference would read `customer_zip_code_prefix` as an integer and drop leading zeros (e.g. `01310` becomes `1310`). It would also leave timestamps as strings if the format was not recognised.
2. Cost: Inference needs an extra full pass over each file before the real read, which would also distort the Part B timings.
3. Stable plans: Fixed types mean Catalyst produces the same logical plan on every run.

With an explicit schema, Spark maps CSV columns by position and ignores the header row. A schema written in the wrong column order would load silently but wrongly, so each file's header is checked against its schema before loading.

### Tables used
Six of the nine Olist files are needed:

| Table | Used for |
|---|---|
| `order_items` | Revenue (`price`) and `product_id`; the central fact table |
| `orders` | Order status, purchase and delivery timestamps |
| `customers` | Customer state |
| `products` | Product category |
| `category_translation` | English category names |
| `order_reviews` | Review scores, for the delivery-delay question |

`order_payments` is deliberately **not** used. Payments are recorded per order, not per item, so joining them to line items would duplicate rows and double-count revenue. `price` from `order_items` is the correct revenue measure. `sellers` and `geolocation` are not needed for the business question.

In [4]:
# ---------------------------------------------------------------
# File locations
# ---------------------------------------------------------------
# CSVs are stored in ./data/ next to this notebook (see data/README.md).
# The path is relative, so it works both on the host machine and inside the Docker container.
DATA_DIR = "data"

# Short table name -> CSV file name
FILES = {
    "orders":               "olist_orders_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "customers":            "olist_customers_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "order_reviews":        "olist_order_reviews_dataset.csv",
}

# Error Message
missing = [f for f in FILES.values() if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing from ./{DATA_DIR}/: {missing}. See data/README.md.")
print("All data files found.")

All data files found.


In [5]:
# ---------------------------------------------------------------
# Explicit schemas
# ---------------------------------------------------------------
# Column ORDER must match each CSV exactly, because Spark maps columns by position when a schema is supplied.
# All fields are nullable: Spark's CSV reader treats every column as
# nullable regardless of the schema, so completeness of key columns is
# checked explicitly later.
# Note: Olist's own header misspells 'length' as 'lenght' in two product columns.
SCHEMAS = {
    "orders": StructType([
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("order_status", StringType(), True),
        StructField("order_purchase_timestamp", TimestampType(), True),
        StructField("order_approved_at", TimestampType(), True),
        StructField("order_delivered_carrier_date", TimestampType(), True),
        StructField("order_delivered_customer_date", TimestampType(), True),
        StructField("order_estimated_delivery_date", TimestampType(), True),
    ]),
    "order_items": StructType([
        StructField("order_id", StringType(), True),
        StructField("order_item_id", IntegerType(), True),
        StructField("product_id", StringType(), True),
        StructField("seller_id", StringType(), True),
        StructField("shipping_limit_date", TimestampType(), True),
        StructField("price", DoubleType(), True),
        StructField("freight_value", DoubleType(), True),
    ]),
    "customers": StructType([
        StructField("customer_id", StringType(), True),
        StructField("customer_unique_id", StringType(), True),
        StructField("customer_zip_code_prefix", StringType(), True),  # string keeps leading zeros
        StructField("customer_city", StringType(), True),
        StructField("customer_state", StringType(), True),
    ]),
    "products": StructType([
        StructField("product_id", StringType(), True),
        StructField("product_category_name", StringType(), True),
        StructField("product_name_lenght", IntegerType(), True),
        StructField("product_description_lenght", IntegerType(), True),
        StructField("product_photos_qty", IntegerType(), True),
        StructField("product_weight_g", DoubleType(), True),
        StructField("product_length_cm", DoubleType(), True),
        StructField("product_height_cm", DoubleType(), True),
        StructField("product_width_cm", DoubleType(), True),
    ]),
    "category_translation": StructType([
        StructField("product_category_name", StringType(), True),
        StructField("product_category_name_english", StringType(), True),
    ]),
    "order_reviews": StructType([
        StructField("review_id", StringType(), True),
        StructField("order_id", StringType(), True),
        StructField("review_score", IntegerType(), True),
        StructField("review_comment_title", StringType(), True),
        StructField("review_comment_message", StringType(), True),
        StructField("review_creation_date", TimestampType(), True),
        StructField("review_answer_timestamp", TimestampType(), True),
    ]),
}

In [6]:
# ---------------------------------------------------------------
# Header validation and CSV loading
# ---------------------------------------------------------------
def check_header(name):
    """Compare the CSV header row with the schema's field names.
    Guards against a schema written in the wrong column order."""
    path = os.path.join(DATA_DIR, FILES[name])
    # Read only the first line of the file as plain text
    header = spark.read.text(path).limit(1).first()[0]
    # Remove quote marks and the invisible byte-order mark (\ufeff)
    # that appears at the start of the translation file
    file_cols = [c.strip().strip('"').lstrip("\ufeff") for c in header.split(",")]
    schema_cols = SCHEMAS[name].fieldNames()
    if file_cols != schema_cols:
        raise ValueError(f"{name}: header {file_cols} does not match schema {schema_cols}")


def load_csv(name, multiline=False):
    """Load one Olist CSV using its explicit schema."""
    check_header(name)
    reader = (
        spark.read
        .option("header", True)                                  # skip the header row
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")        # Olist timestamp format
        .option("mode", "PERMISSIVE")                            # bad values -> null 
    )
    if multiline:
        # Review comments contain line breaks and quote marks inside
        # quoted fields. Without these options each line break would be
        # read as a new row, corrupting the table.
        reader = reader.option("multiLine", True).option("escape", '"')
    return reader.schema(SCHEMAS[name]).csv(os.path.join(DATA_DIR, FILES[name]))


# Load each table.
# No data is read until an action (count, show, collect) is called.
orders_df               = load_csv("orders")
order_items_df          = load_csv("order_items")
customers_df            = load_csv("customers")
products_df             = load_csv("products")
category_translation_df = load_csv("category_translation")
order_reviews_df        = load_csv("order_reviews", multiline=True)

# Keep references in one place for the summary checks below
tables = {
    "orders": orders_df,
    "order_items": order_items_df,
    "customers": customers_df,
    "products": products_df,
    "category_translation": category_translation_df,
    "order_reviews": order_reviews_df,
}
print("Headers validated and all tables loaded.")

Headers validated and all tables loaded.


## Initial Exploration

In [7]:
# Confirm the schemas were applied (types, not all strings)
order_items_df.printSchema()
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [8]:
# ---------------------------------------------------------------
# Row counts
# ---------------------------------------------------------------
rows = [(name, df.count(),  len(df.columns)) for name, df in tables.items()]
pd.DataFrame(rows, columns=["Table", "Rows loaded", "Columns"])

,Table,Rows loaded,Columns
0,orders,99441,8
1,order_items,112650,7
2,customers,99441,5
3,products,32951,9
4,category_translation,71,2
5,order_reviews,99224,7


In [9]:
# ---------------------------------------------------------------
# Null counts per column 
# ---------------------------------------------------------------
def null_profile(name, df):
    # One pass over the table: for every column, sum 1 where the value is null (only columns with at least one null are shown)
    counts = df.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    ).first()
    return [(name, c, counts[c]) for c in df.columns if counts[c]]

profile = [r for name, df in tables.items() for r in null_profile(name, df)]
pd.DataFrame(profile, columns=["Table", "Column", "Null count"]) if profile else "No nulls found."

,Table,Column,Null count
0,orders,order_approved_at,160
1,orders,order_delivered_carrier_date,1783
2,orders,order_delivered_customer_date,2965
3,products,product_category_name,610
4,products,product_name_lenght,610
5,products,product_description_lenght,610
6,products,product_photos_qty,610
7,products,product_weight_g,2
8,products,product_length_cm,2
9,products,product_height_cm,2


In [10]:
# ---------------------------------------------------------------
# Order status distribution
# ---------------------------------------------------------------
# Note: delivery performance analysis done on only delivered orders.
n_orders = orders_df.count()

(orders_df
 .groupBy("order_status")
 .count()
 .withColumn("pct", F.round(100 * F.col("count") / n_orders, 2))
 .orderBy(F.desc("count"))
 .show())

+------------+-----+-----+
|order_status|count|  pct|
+------------+-----+-----+
|   delivered|96478|97.02|
|     shipped| 1107| 1.11|
|    canceled|  625| 0.63|
| unavailable|  609| 0.61|
|    invoiced|  314| 0.32|
|  processing|  301|  0.3|
|     created|    5| 0.01|
|    approved|    2|  0.0|
+------------+-----+-----+



In [11]:
# ---------------------------------------------------------------
# Orders per quarter and overall date range
# ---------------------------------------------------------------
# Shows whether the first and last quarters have enough orders
# to be ranked fairly for Part A.
(orders_df
 .withColumn("year", F.year("order_purchase_timestamp"))
 .withColumn("quarter", F.quarter("order_purchase_timestamp"))
 .groupBy("year", "quarter")
 .count()
 .orderBy("year", "quarter")
 .show(20))

orders_df.select(
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase")
).show()

+----+-------+-----+
|year|quarter|count|
+----+-------+-----+
|2016|      3|    4|
|2016|      4|  325|
|2017|      1| 5262|
|2017|      2| 9349|
|2017|      3|12642|
|2017|      4|17848|
|2018|      1|21208|
|2018|      2|19979|
|2018|      3|12820|
|2018|      4|    4|
+----+-------+-----+

+-------------------+-------------------+
|     first_purchase|      last_purchase|
+-------------------+-------------------+
|2016-09-04 21:15:19|2018-10-17 17:30:18|
+-------------------+-------------------+



In [12]:
# ---------------------------------------------------------------
# Customer distribution by state
# ---------------------------------------------------------------
# A low-cardinality dimension (27 states) that is naturally skewed
# towards São Paulo (SP). Useful for Part B.
n_customers = customers_df.count()

(customers_df
 .groupBy("customer_state")
 .count()
 .withColumn("pct", F.round(100 * F.col("count") / n_customers, 2))
 .orderBy(F.desc("count"))
 .show(27))

+--------------+-----+-----+
|customer_state|count|  pct|
+--------------+-----+-----+
|            SP|41746|41.98|
|            RJ|12852|12.92|
|            MG|11635| 11.7|
|            RS| 5466|  5.5|
|            PR| 5045| 5.07|
|            SC| 3637| 3.66|
|            BA| 3380|  3.4|
|            DF| 2140| 2.15|
|            ES| 2033| 2.04|
|            GO| 2020| 2.03|
|            PE| 1652| 1.66|
|            CE| 1336| 1.34|
|            PA|  975| 0.98|
|            MT|  907| 0.91|
|            MA|  747| 0.75|
|            MS|  715| 0.72|
|            PB|  536| 0.54|
|            PI|  495|  0.5|
|            RN|  485| 0.49|
|            AL|  413| 0.42|
|            SE|  350| 0.35|
|            TO|  280| 0.28|
|            RO|  253| 0.25|
|            AM|  148| 0.15|
|            AC|   81| 0.08|
|            AP|   68| 0.07|
|            RR|   46| 0.05|
+--------------+-----+-----+



In [13]:
# ---------------------------------------------------------------
# Data-quality checks
# ---------------------------------------------------------------
# Investigate how many reviews each order has.
reviews_per_order = order_reviews_df.groupBy("order_id").count()

# Investigate categories that exist in products but have
# NO match in the translation table
untranslated = (products_df
                .filter(F.col("product_category_name").isNotNull())
                .select("product_category_name").distinct()
                .join(category_translation_df, "product_category_name", "left_anti"))

# No. orders delivered 
delivered = orders_df.filter(F.col("order_status") == "delivered")

checks = [
    ("Orders with more than one review",
     reviews_per_order.filter("count > 1").count()),
    ("Duplicate review_id values",
     order_reviews_df.count() - order_reviews_df.select("review_id").distinct().count()),
    ("Products with no category",
     products_df.filter(F.col("product_category_name").isNull()).count()),
    ("Categories missing from translation table",
     untranslated.count()),
    ("Delivered orders missing a delivery date",
     delivered.filter(F.col("order_delivered_customer_date").isNull()).count()),
    ("Order items whose order_id is not in orders",
     order_items_df.join(orders_df, "order_id", "left_anti").count()),
]
pd.DataFrame(checks, columns=["Check", "Count"])

,Check,Count
0,Orders with more than one review,547
1,Duplicate review_id values,814
2,Products with no category,610
3,Categories missing from translation table,2
4,Delivered orders missing a delivery date,8
5,Order items whose order_id is not in orders,0


In [14]:
# Categories have no English name
print("Categories with no English translation:")
untranslated.show(truncate=False)

Categories with no English translation:
+---------------------------------------------+
|product_category_name                        |
+---------------------------------------------+
|pc_gamer                                     |
|portateis_cozinha_e_preparadores_de_alimentos|
+---------------------------------------------+



In [15]:
# ---------------------------------------------------------------
# Rows per key: Determine concentrated is each candidate key
# ---------------------------------------------------------------
def key_frequency_summary(df, key):
    freq = df.groupBy(key).count()          # rows per distinct key value
    total = df.count()
    n_keys = freq.count()
    top_1pct = max(1, n_keys // 100)        # number of keys in the top 1%
    top_share = (freq.orderBy(F.desc("count"))
                     .limit(top_1pct)
                     .agg(F.sum("count"))
                     .first()[0])
    # Exact percentiles of rows-per-key (0.0 = no approximation error)
    q = freq.approxQuantile("count", [0.5, 0.9, 0.99], 0.0)
    stats = freq.agg(F.max("count"), F.avg("count")).first()
    return {
        "Key": key,
        "Rows": total,
        "Distinct keys": n_keys,
        "Mean rows/key": round(stats[1], 2),
        "Median": q[0], "P90": q[1], "P99": q[2],
        "Max rows/key": stats[0],
        "Rows held by top 1% of keys (%)": round(100 * top_share / total, 2),
    }

pd.DataFrame([key_frequency_summary(order_items_df, "product_id"),
              key_frequency_summary(order_items_df, "order_id")]).set_index("Key").T

Key,product_id,order_id
Rows,112650.00,112650.00
Distinct keys,32951.00,98666.00
Mean rows/key,3.42,1.14
Median,1.00,1.00
P90,6.00,1.00
P99,33.00,3.00
Max rows/key,527.00,21.00
Rows held by top 1% of keys (%),22.09,4.31


In [16]:
# The ten best-selling products: the 'heavy' keys behind any skew
(order_items_df
 .groupBy("product_id")
 .count()
 .orderBy(F.desc("count"))
 .show(10, truncate=False))

+--------------------------------+-----+
|product_id                      |count|
+--------------------------------+-----+
|aca2eb7d00ea1a7b8ebd4e68314663af|527  |
|99a4788cb24856965c36a24e339b6058|488  |
|422879e10f46682990de24d770e7f83d|484  |
|389d119b48cf3043d311335e499d9c6b|392  |
|368c6c730842d78016ad823897a372db|388  |
|53759a2ecddad2bb87a079a1f1519f73|373  |
|d1c427060a0f73f6b889a5c7c61f2ac4|343  |
|53b36df67ebb7c41585e8d54d6772e08|323  |
|154e7e31ebfa092203795c972e5804a6|281  |
|3dd2a17168ec895c781a9191c1e95ad7|274  |
+--------------------------------+-----+
only showing top 10 rows


# Part A 

## A.1 Business query

### Business question
**For each Brazilian customer state and quarter (2017 Q1 – 2018 Q3), which three product categories generate the most revenue? Across categories and regions, do late deliveries correlate with lower review scores? And which top-revenue category–state combinations appear to be held back by logistics rather than by demand?**

The question comes directly from the approved proposal and has three linked parts:

| Part | Question | Output |
|---|---|---|
| 1. Revenue ranking | Which categories lead revenue in each state and quarter, among categories with enough orders to be meaningful? | `top_categories` |
| 2. Delivery vs satisfaction | Is lateness (actual − estimated delivery date) associated with lower review scores, and does this differ by category or by state? | `logistics_by_category`, `logistics_by_state` |
| 3. Logistics vs demand | Which top-3 category–state combinations have an above-average late rate *and* an above-average review penalty for lateness? | `top_with_logistics` |

### Analytical pipeline and required operations

| Operation (from the brief) | Where it is used and why it is needed |
|---|---|
| **Joins** (broadcast where appropriate) | Six-table join to build one enriched line-item table. `products` and `category_translation` are small dimension tables and are broadcast. Aggregated state–quarter totals and the one-row overall baseline are also broadcast into later joins. |
| **Multi-level / nested aggregation** | Revenue is grouped by state × quarter × category. The review analysis is a *two-stage* aggregation: line items are first reduced to one row per order (so an order with three items is not counted three times), then aggregated per category or state. |
| **Window functions** | `rank()` of revenue within each (state, quarter) partition. The top-3 filter depends on the rank, so it cannot be expressed with a plain `GROUP BY`. |
| **Post-aggregation filter (HAVING)** | Groups are kept only if they have at least a minimum number of distinct orders. This must happen *after* aggregation because the order count only exists once the group is formed. |
| **Time-based analysis** | Quarter derived from `order_purchase_timestamp`; delivery delay in days derived from two timestamps. |

### Why a single GROUP BY is insufficient
- **Ranking within a group** (top 3 per state and quarter) needs a window partitioned by state and quarter over already-aggregated revenue. That is an aggregation of an aggregation.
- **Late vs on-time review comparison** needs conditional aggregates computed over order-level rows. A single `GROUP BY` over line items would weight multi-item orders more heavily. Section 1 found 547 orders with more than one review, so reviews must also be collapsed to one score per order *before* joining, or those orders' revenue would be double-counted.
- **Volume thresholds** can only be applied after the counts exist (HAVING).
- **Combining demand and logistics** requires joining two independently aggregated results at different grains.

### Why distributed processing is appropriate
The data volume is modest: about 110k line items (a few MB), which would fit in memory on one machine. The justification is therefore not size alone, but the *shape of the workload*:

- **Every expensive step is shuffle-bound.** These include the items ⋈ orders join on `order_id` (≈112k × ≈97k rows), grouping on a composite key, the window partitioned by (state, quarter), and the per-order deduplication. Each requires rows with the same key to be co-located, which is exactly what Spark's partitioned shuffle parallelises across cores/nodes.
- **The data is skewed in ways that affect parallel work.** SP holds 42% of customers, and the top 1% of products hold 22% of line items. How work is split across partitions therefore matters; Part B investigates this.
- **Intermediate results are reused.** One enriched table feeds several downstream aggregations. Spark's lazy DAG and caching let it be computed once.
- **The code scales without rewriting.** Olist is a two-year sample of one marketplace. The same pipeline would run unchanged over many years or many marketplaces, where a single-machine approach would run out of memory.

## A.2 DataFrame implementation

### A.2.1 Query parameters

In [17]:
# ---------------------------------------------------------------
# Query parameters (shared by the DataFrame and Spark SQL versions)
# ---------------------------------------------------------------
# Date range reduced from 10 quarters to 7 quarters as
# 2016 Q3 (4 orders), 2016 Q4 (325) and 2018 Q4 (4) are too sparse to rank fairly

START_TS = "2017-01-01 00:00:00"      # inclusive: start of 2017 Q1
END_TS   = "2018-10-01 00:00:00"      # exclusive: end of 2018 Q3

# Orders with no completed sale are excluded from revenue
EXCLUDED_STATUSES = ["canceled", "unavailable"]

TOP_N = 3                             # categories kept per (state, quarter)

# Minimum-volume thresholds (HAVING). Final values are chosen from A.2.3
# from the sensitivity check.
MIN_GROUP_ORDERS   = 10     # orders per (state, quarter, category) group
MIN_SEGMENT_ORDERS = 200    # orders per category / state for the review analysis
MIN_LATE_ORDERS    = 30     # late orders needed for a reliable 'late' average

### A.2.2 Base Table 

This step builds one row per order line item, with everything the later analyses need.

1. **Reviews are pre-aggregated per order** (mean score). This removes the 547 multi-review orders as a source of row duplication before any join.
2. **`orders` is filtered and projected first**: date range and status, keeping only the six columns needed.
3. **Joins, in this order:**
   - `order_items` ⋈ filtered `orders` (inner)
   - ⋈ `customers` (inner; Section 1.6 found 0 orphans)
   - ⋈ **broadcast** `products` (**left**, so the 610 uncategorised products are kept)
   - ⋈ **broadcast** `category_translation` (**left**, so the 2 untranslated categories are kept)
   - ⋈ per-order reviews (**left**)
4. **Derived columns:**
   - `year_quarter` (e.g. `2017-Q3`, which sorts correctly as text)
   - `category`: English name, falling back to the Portuguese name, then `uncategorised`
   - `delay_days` (actual − estimated delivery date, in whole days; positive = late)
   - `is_late`

**Caching decision.** `base_df` is cached because it is the root of every downstream result:
- the threshold check
- the revenue groups
- the state–quarter totals
- four review aggregations (overall, category, state, state × category)
- the final combined table

Without caching, each of these actions would re-execute the full lineage: parsing six CSVs, the multi-line review file, and five joins. The DAG would contain the same scan-and-join subtree once per action. With `cache()`, the first action materialises it in memory, and later jobs start from an `InMemoryTableScan`. At ~112k narrow rows the cached table is a few MB, well within the 4 GB driver. Downstream aggregates are *not* cached, because each is used only once or twice and is cheap to recompute from the cached base.

In [18]:
# ---------------------------------------------------------------
# Step 1: one review score per order (removes multi-review duplication)
# ---------------------------------------------------------------
reviews_per_order_df = (
    order_reviews_df
    .groupBy("order_id")
    .agg(F.avg("review_score").alias("review_score"))
)

# ---------------------------------------------------------------
# Step 2: filter and project orders BEFORE joining
# ---------------------------------------------------------------
filtered_orders_df = (
    orders_df
    .filter(
        (F.col("order_purchase_timestamp") >= F.to_timestamp(F.lit(START_TS))) &
        (F.col("order_purchase_timestamp") <  F.to_timestamp(F.lit(END_TS))) &
        (~F.col("order_status").isin(EXCLUDED_STATUSES))
    )
    .select("order_id", "customer_id", "order_status", "order_purchase_timestamp",
            "order_delivered_customer_date", "order_estimated_delivery_date")
)

# ---------------------------------------------------------------
# Step 3 + 4: joins and derived columns -> enriched line-item table
# ---------------------------------------------------------------
base_df = (
    order_items_df
    .select("order_id", "product_id", "price")                       # projection
    .join(filtered_orders_df, "order_id")                             # fact x fact (inner)
    .join(customers_df.select("customer_id", "customer_state"), "customer_id")
    .join(F.broadcast(products_df.select("product_id", "product_category_name")),
          "product_id", "left")                                       # small dimension
    .join(F.broadcast(category_translation_df), "product_category_name", "left")
    .join(reviews_per_order_df, "order_id", "left")
    # --- time-based derived columns ---
    .withColumn("year_quarter",
                F.format_string("%d-Q%d",
                                F.year("order_purchase_timestamp"),
                                F.quarter("order_purchase_timestamp")))
    .withColumn("category",
                F.coalesce("product_category_name_english",
                           "product_category_name",
                           F.lit("uncategorised")))
    # positive = delivered late; null for non-delivered orders or missing dates
    .withColumn("delay_days",
                F.when(F.col("order_status") == "delivered",
                       F.datediff(F.to_date("order_delivered_customer_date"),
                                  F.to_date("order_estimated_delivery_date"))))
    .withColumn("is_late", F.col("delay_days") > 0)
    .select("order_id", "customer_state", "year_quarter", "category", "price",
            "order_status", "delay_days", "is_late", "review_score")
    .cache()
)

# First action materialises the cache
n_base = base_df.count()
print(f"Enriched base table: {n_base:,} line items cached")
base_df.show(5, truncate=False)

Enriched base table: 111,753 line items cached
+--------------------------------+--------------+------------+----------+-----+------------+----------+-------+------------+
|order_id                        |customer_state|year_quarter|category  |price|order_status|delay_days|is_late|review_score|
+--------------------------------+--------------+------------+----------+-----+------------+----------+-------+------------+
|e481f51cbdc54678b7cc49136f2d6af7|SP            |2017-Q4     |housewares|29.99|delivered   |-8        |false  |4.0         |
|53cdb2fc8bc7dce0b6741e2150273451|BA            |2018-Q3     |perfumery |118.7|delivered   |-6        |false  |4.0         |
|47770eb9100c2d0c44946d9cf07ec65d|GO            |2018-Q3     |auto      |159.9|delivered   |-18       |false  |5.0         |
|949d5b44dbf5de918fe9c16f97b45f8a|RN            |2017-Q4     |pet_shop  |45.0 |delivered   |-13       |false  |5.0         |
|ad21c59c0840e6cb83a9ceb5573f8159|SP            |2018-Q1     |stationery|19.9 

### A.2.3 Choosing the minimum-volume thresholds
The thresholds are chosen from the data. The tables below show, for several candidate values:
- how many groups survive
- how much revenue they cover
- how many (state, quarter) partitions still have a full top 3

The same is shown for the category and state-level review analysis.

In [19]:
# ---------------------------------------------------------------
# Sensitivity of the revenue-ranking threshold (MIN_GROUP_ORDERS)
# ---------------------------------------------------------------
# The grouped result is small (at most ~27 states x 7 quarters x ~74
# categories), so it is collected to pandas for the what-if table.
groups_pd = (
    base_df
    .groupBy("customer_state", "year_quarter", "category")
    .agg(F.countDistinct("order_id").alias("n_orders"),
         F.sum("price").alias("revenue"))
    .toPandas()
)

total_rev = groups_pd["revenue"].sum()
n_sq = groups_pd[["customer_state", "year_quarter"]].drop_duplicates().shape[0]
rows = []
for t in [1, 5, 10, 20, 30, 50, 100]:
    kept = groups_pd[groups_pd["n_orders"] >= t]
    per_sq = kept.groupby(["customer_state", "year_quarter"]).size()
    rows.append({
        "Min orders": t,
        "Groups kept": len(kept),
        "Groups kept (%)": round(100 * len(kept) / len(groups_pd), 1),
        "Revenue covered (%)": round(100 * kept["revenue"].sum() / total_rev, 1),
        "States with any group": kept["customer_state"].nunique(),
        f"(state, quarter) with full top {TOP_N}": int((per_sq >= TOP_N).sum()),
    })
print(f"{len(groups_pd):,} (state, quarter, category) groups across {n_sq} (state, quarter) pairs")
pd.DataFrame(rows)

5,898 (state, quarter, category) groups across 189 (state, quarter) pairs


,Min orders,Groups kept,Groups kept (%),Revenue covered (%),States with any group,"(state, quarter) with full top 3"
0,1,5898,100.0,100.0,27,188
1,5,2469,41.9,91.1,24,133
2,10,1600,27.1,84.2,21,97
3,20,924,15.7,73.7,15,64
4,30,672,11.4,67.1,12,51
5,50,395,6.7,56.4,9,34
6,100,190,3.2,42.7,5,17


In [20]:
# ---------------------------------------------------------------
# Sensitivity of the review-analysis threshold (MIN_SEGMENT_ORDERS)
# ---------------------------------------------------------------
# Order-level rows: delivered orders with a delivery date and a review
seg_pd = (
    base_df
    .filter(F.col("delay_days").isNotNull() & F.col("review_score").isNotNull())
    .select("order_id", "customer_state", "category", F.col("is_late").cast("int").alias("late"))
    .distinct()
    .toPandas()
)

rows = []
for dim in ["category", "customer_state"]:
    per_seg = seg_pd.groupby(dim).agg(n_orders=("order_id", "nunique"), n_late=("late", "sum"))
    total_orders = seg_pd["order_id"].nunique()
    for t in [50, 100, 200, 500, 1000]:
        kept = per_seg[(per_seg["n_orders"] >= t) & (per_seg["n_late"] >= MIN_LATE_ORDERS)]
        rows.append({
            "Dimension": dim, "Min orders": t,
            "Segments kept": f"{len(kept)} / {len(per_seg)}",
            "Orders covered (%)": round(100 * kept["n_orders"].sum() / per_seg["n_orders"].sum(), 1),
        })
pd.DataFrame(rows)

,Dimension,Min orders,Segments kept,Orders covered (%)
0,category,50,29 / 74,93.4
1,category,100,29 / 74,93.4
2,category,200,29 / 74,93.4
3,category,500,25 / 74,91.7
4,category,1000,22 / 74,89.6
5,customer_state,50,21 / 27,99.1
6,customer_state,100,21 / 27,99.1
7,customer_state,200,21 / 27,99.1
8,customer_state,500,17 / 27,97.4
9,customer_state,1000,12 / 27,93.5


**Threshold Choice**

- `MIN_GROUP_ORDERS = 10`. This removes the long tail of tiny groups (73% of the 5,898 groups), in which one or two expensive orders could decide a ranking. It still keeps 84.2% of revenue, 21 of 27 states, and a full top 3 for 97 of the 189 (state, quarter) pairs.
  - Raising it to 20 would cost a further 10 points of revenue, remove 6 more states, and cut full top-3 pairs by a third.
  - Lowering it to 5 would add only 7 points of revenue while admitting groups too small to rank reliably.
- `MIN_LATE_ORDERS = 30`. This is the binding constraint for the review analysis. Category results are identical at 50, 100 and 200 minimum orders (29 of 74 kept), showing that the late-order requirement, not total volume, determines which segments qualify. Thirty is a common minimum for a stable sample mean; below it, the average review for late orders would depend on only a handful of customers.
- `MIN_SEGMENT_ORDERS = 200`.This is a safeguard that is not currently binding. Raising it to 500 would drop 4 categories and 4 states without improving reliability.
- Coverage: The kept segments cover 93.4% of eligible orders by category and 99.1% by state, so the excluded segments are genuinely marginal.

### A.2.4 Analysis 1: Top revenue categories per state and quarter

Steps:
1. Aggregate revenue, distinct orders and line items per (state, quarter, category).
2. Apply the **HAVING**-equivalent `filter` on distinct orders.
3. Join a broadcast table of state–quarter revenue totals, so each category's share of its state's quarterly revenue can be computed.

   The total is taken before the threshold filter, so the share is relative to all revenue in that state and quarter.
4. **Rank** within each (state, quarter) with a window function and keep the top 3.

Ranking on rounded revenue-  Revenue is rounded to cents before ranking. Floating-point sums can differ in the last digits depending on the order in which partitions are combined. Categories with genuinely equal revenue (common in small states, where a group may be a few identical-price items) could otherwise be ranked differently from run to run. `category` is added as a deterministic tie-breaker, so `rank()` never produces ties and the result is reproducible.

In [21]:
# ---------------------------------------------------------------
# Revenue totals per (state, quarter), before any threshold
# ---------------------------------------------------------------
state_quarter_totals_df = (
    base_df
    .groupBy("customer_state", "year_quarter")
    .agg(F.sum("price").alias("state_quarter_revenue"))
)

# Window: one partition per (state, quarter), highest revenue first
revenue_rank_window = (
    Window
    .partitionBy("customer_state", "year_quarter")
    .orderBy(F.desc("revenue"), F.asc("category"))   # category = tie-breaker
)

top_categories_df = (
    base_df
    .groupBy("customer_state", "year_quarter", "category")
    .agg(F.round(F.sum("price"), 2).alias("revenue"),
         F.countDistinct("order_id").alias("n_orders"),
         F.count("*").alias("n_items"))
    .filter(F.col("n_orders") >= MIN_GROUP_ORDERS)                  # HAVING
    .join(F.broadcast(state_quarter_totals_df), ["customer_state", "year_quarter"])
    .withColumn("revenue_share_pct",
                F.round(100 * F.col("revenue") / F.col("state_quarter_revenue"), 2))
    .withColumn("revenue_rank", F.rank().over(revenue_rank_window))  # window function
    .filter(F.col("revenue_rank") <= TOP_N)
    .select("customer_state", "year_quarter", "revenue_rank", "category",
            "revenue", "revenue_share_pct", "n_orders", "n_items")
)

In [22]:
# Display: the full result is several hundred rows,show one state in full
# (SP, the largest market) and summarise the rest.
print(f"Top-{TOP_N} rows across all states and quarters: {top_categories_df.count()}")
(top_categories_df
 .filter(F.col("customer_state") == "SP")
 .orderBy("year_quarter", "revenue_rank")
 .show(3 * 7, truncate=False))

Top-3 rows across all states and quarters: 324
+--------------+------------+------------+---------------------+---------+-----------------+--------+-------+
|customer_state|year_quarter|revenue_rank|category             |revenue  |revenue_share_pct|n_orders|n_items|
+--------------+------------+------------+---------------------+---------+-----------------+--------+-------+
|SP            |2017-Q1     |1           |furniture_decor      |23819.56 |9.38             |274     |340    |
|SP            |2017-Q1     |2           |sports_leisure       |23287.74 |9.17             |144     |175    |
|SP            |2017-Q1     |3           |bed_bath_table       |19370.73 |7.63             |189     |215    |
|SP            |2017-Q2     |1           |bed_bath_table       |40435.78 |8.46             |397     |453    |
|SP            |2017-Q2     |2           |computers_accessories|33516.35 |7.01             |218     |257    |
|SP            |2017-Q2     |3           |cool_stuff           |32498.33 

In [28]:
# #1 revenue category for every (state, quarter):
leaders_pd = (top_categories_df
              .filter(F.col("revenue_rank") == 1)
              .select("customer_state", "year_quarter", "category")
              .toPandas()
              .pivot(index="customer_state", columns="year_quarter", values="category")
              .fillna("-"))   # '-' = no category met the minimum-orders threshold

# Order states by market size (number of top-1 slots filled, then name)
leaders_pd = leaders_pd.loc[
    leaders_pd.apply(lambda r: (r != "-").sum(), axis=1)
              .sort_values(ascending=False, kind="stable").index]
leaders_pd

year_quarter,2017-Q1,2017-Q2,2017-Q3,2017-Q4,2018-Q1,2018-Q2,2018-Q3
customer_state,,,,,,,
BA,health_beauty,computers_accessories,health_beauty,watches_gifts,sports_leisure,watches_gifts,health_beauty
DF,sports_leisure,sports_leisure,health_beauty,watches_gifts,watches_gifts,health_beauty,watches_gifts
ES,sports_leisure,health_beauty,bed_bath_table,watches_gifts,computers_accessories,watches_gifts,watches_gifts
GO,cool_stuff,cool_stuff,bed_bath_table,watches_gifts,health_beauty,watches_gifts,health_beauty
MG,furniture_decor,cool_stuff,bed_bath_table,watches_gifts,health_beauty,health_beauty,health_beauty
PR,cool_stuff,sports_leisure,sports_leisure,furniture_decor,computers_accessories,watches_gifts,watches_gifts
RJ,furniture_decor,health_beauty,bed_bath_table,bed_bath_table,watches_gifts,watches_gifts,health_beauty
RS,health_beauty,bed_bath_table,bed_bath_table,computers_accessories,sports_leisure,bed_bath_table,health_beauty
SC,garden_tools,computers_accessories,health_beauty,sports_leisure,sports_leisure,furniture_decor,health_beauty


In [23]:
# Which categories dominate the top-3 slots nationally?
(top_categories_df
 .groupBy("category")
 .agg(F.count("*").alias("top3_appearances"),
      F.sum(F.when(F.col("revenue_rank") == 1, 1).otherwise(0)).alias("times_ranked_1st"),
      F.countDistinct("customer_state").alias("n_states"))
 .orderBy(F.desc("top3_appearances"), F.asc("category"))
 .show(10, truncate=False))

+---------------------+----------------+----------------+--------+
|category             |top3_appearances|times_ranked_1st|n_states|
+---------------------+----------------+----------------+--------+
|health_beauty        |84              |49              |21      |
|watches_gifts        |59              |27              |19      |
|sports_leisure       |45              |12              |17      |
|bed_bath_table       |36              |11              |14      |
|computers_accessories|31              |9               |18      |
|cool_stuff           |17              |7               |10      |
|furniture_decor      |17              |6               |11      |
|housewares           |9               |0               |7       |
|auto                 |8               |0               |6       |
|telephony            |6               |0               |6       |
+---------------------+----------------+----------------+--------+
only showing top 10 rows


### A.2.5 Analysis 2: does late delivery correlate with lower review scores?

This is a nested aggregation**:

1. Stage 1: reduce to order level - Delay and review score are order-level facts repeated on each line item. `distinct()` keeps one row per order within each segment. Without it, an order with three items in a category would count three times.
2. Stage 2: aggregate per segment, computing:
   - order count and late count
   - late rate and mean delay
   - mean review for on-time and for late orders
   - the **review gap** (on-time − late)
   - the Pearson correlation between `delay_days` and `review_score`

The same function is applied at four levels: overall (the baseline), per category, per state, and per state × category. The HAVING-equivalent filter keeps only segments with enough orders and enough late orders for the late-order mean to be meaningful.

Correlation is deliberately not computed per state × quarter × category. Many of those groups contain only a handful of orders and correlations from  small samples are unreliable.

In [24]:
# Order-level rows eligible
review_base_df = base_df.filter(
    F.col("delay_days").isNotNull() & F.col("review_score").isNotNull()
)

def logistics_metrics(dims):
    # Two-stage aggregation of delivery performance vs review score.
    # dims = [] gives the overall baseline (a single row).
    late = F.col("is_late")
    return (
        review_base_df
        # Stage 1: one row per order within each segment
        .select("order_id", *dims, "delay_days", "is_late", "review_score")
        .distinct()
        # Stage 2: aggregate per segment
        .groupBy(*dims)
        .agg(
            F.count("*").alias("n_orders"),
            F.sum(late.cast("int")).alias("n_late"),
            F.round(100 * F.avg(late.cast("int")), 2).alias("late_rate_pct"),
            F.round(F.avg("delay_days"), 2).alias("avg_delay_days"),
            F.round(F.avg(F.when(~late, F.col("review_score"))), 3).alias("avg_review_on_time"),
            F.round(F.avg(F.when(late, F.col("review_score"))), 3).alias("avg_review_late"),
            F.round(F.avg(F.when(~late, F.col("review_score")))
                    - F.avg(F.when(late, F.col("review_score"))), 3).alias("review_gap"),
            F.round(F.try_divide(
                F.covar_samp(F.col("delay_days").cast("double"), F.col("review_score")),
                F.stddev_samp(F.col("delay_days").cast("double")) * F.stddev_samp("review_score")
            ), 3).alias("corr_delay_review"),
        )
        # HAVING-equivalent: enough orders, and enough late orders
        .filter((F.col("n_orders") >= MIN_SEGMENT_ORDERS) &
                (F.col("n_late") >= MIN_LATE_ORDERS))
    )

overall_logistics_df        = logistics_metrics([])
logistics_by_category_df    = logistics_metrics(["category"])
logistics_by_state_df       = logistics_metrics(["customer_state"])
logistics_by_state_cat_df   = logistics_metrics(["customer_state", "category"])

print("Overall baseline (all eligible delivered orders):")
overall_logistics_df.show(truncate=False)

Overall baseline (all eligible delivered orders):
+--------+------+-------------+--------------+------------------+---------------+----------+-----------------+
|n_orders|n_late|late_rate_pct|avg_delay_days|avg_review_on_time|avg_review_late|review_gap|corr_delay_review|
+--------+------+-------------+--------------+------------------+---------------+----------+-----------------+
|95560   |6378  |6.67         |-11.84        |4.291             |2.272          |2.019     |-0.27            |
+--------+------+-------------+--------------+------------------+---------------+----------+-----------------+



In [25]:
print("Categories where lateness costs the most review points (top 10):")
(logistics_by_category_df
 .orderBy(F.desc("review_gap"), F.asc("category"))
 .limit(10)
 .show(truncate=False))

print("States, ordered by late-delivery rate:")
(logistics_by_state_df
 .orderBy(F.desc("late_rate_pct"), F.asc("customer_state"))
 .show(27, truncate=False))

Categories where lateness costs the most review points (top 10):
+-------------------------------+--------+------+-------------+--------------+------------------+---------------+----------+-----------------+
|category                       |n_orders|n_late|late_rate_pct|avg_delay_days|avg_review_on_time|avg_review_late|review_gap|corr_delay_review|
+-------------------------------+--------+------+-------------+--------------+------------------+---------------+----------+-----------------+
|musical_instruments            |605     |46    |7.6          |-11.46        |4.426             |1.913          |2.513     |-0.351           |
|audio                          |343     |40    |11.66        |-9.87         |4.111             |1.725          |2.386     |-0.408           |
|toys                           |3752    |232   |6.18         |-12.02        |4.378             |2.136          |2.242     |-0.338           |
|sports_leisure                 |7468    |487   |6.52         |-11.82        

### A.2.6 Analysis 3: logistics issue or demand issue?

Each top-3 (state, quarter, category) row is joined to the delivery metrics for state × category. The state × category metrics are pooled over all seven quarters, to give enough orders for a reliable rate.

A row is flagged `logistics concern` when both conditions hold:
- its late rate is above the national baseline
- its review gap is above the national baseline

In other words, deliveries are late more often than usual and lateness costs more review points than usual. The one-row baseline is attached with a broadcast cross join, so the whole comparison stays inside the DataFrame plan.

Rows whose state × category combination has too few orders for a reliable estimate are labelled `insufficient data`.

In [26]:
baseline_df = overall_logistics_df.select(
    F.col("late_rate_pct").alias("baseline_late_rate_pct"),
    F.col("review_gap").alias("baseline_review_gap"),
)

top_with_logistics_df = (
    top_categories_df
    .join(logistics_by_state_cat_df.select(
              "customer_state", "category",
              F.col("n_orders").alias("logistics_orders"),
              "late_rate_pct", "review_gap"),
          ["customer_state", "category"], "left")
    .crossJoin(F.broadcast(baseline_df))
    .withColumn(
        "logistics_flag",
        F.when(F.col("late_rate_pct").isNull(), "insufficient data")
         .when((F.col("late_rate_pct") > F.col("baseline_late_rate_pct")) &
               (F.col("review_gap") > F.col("baseline_review_gap")), "logistics concern")
         .otherwise("no concern"))
    .select("customer_state", "year_quarter", "revenue_rank", "category", "revenue",
            "revenue_share_pct", "logistics_orders", "late_rate_pct", "review_gap",
            "logistics_flag")
)

print("How many top-3 slots are affected by logistics?")
(top_with_logistics_df
 .groupBy("logistics_flag")
 .count()
 .orderBy(F.desc("count"))
 .show())

How many top-3 slots are affected by logistics?
+-----------------+-----+
|   logistics_flag|count|
+-----------------+-----+
|insufficient data|  258|
|       no concern|   42|
|logistics concern|   24|
+-----------------+-----+



In [27]:
print("Top-revenue category-state combinations flagged as a logistics concern:")
(top_with_logistics_df
 .filter(F.col("logistics_flag") == "logistics concern")
 .groupBy("customer_state", "category")
 .agg(F.count("*").alias("quarters_in_top3"),
      F.round(F.sum("revenue"), 2).alias("top3_revenue"),
      F.first("late_rate_pct").alias("late_rate_pct"),
      F.first("review_gap").alias("review_gap"))
 .orderBy(F.desc("top3_revenue"))
 .show(15, truncate=False))

Top-revenue category-state combinations flagged as a logistics concern:
+--------------+---------------------+----------------+------------+-------------+----------+
|customer_state|category             |quarters_in_top3|top3_revenue|late_rate_pct|review_gap|
+--------------+---------------------+----------------+------------+-------------+----------+
|RJ            |watches_gifts        |6               |162832.27   |13.72        |2.091     |
|RJ            |health_beauty        |5               |97623.47    |8.74         |2.327     |
|RJ            |bed_bath_table       |3               |87221.28    |14.75        |2.12      |
|RJ            |computers_accessories|2               |55517.77    |11.89        |2.498     |
|RJ            |sports_leisure       |2               |43943.67    |13.84        |2.395     |
|RS            |bed_bath_table       |3               |33087.57    |8.43         |2.061     |
|RJ            |toys                 |1               |29546.79    |10.61        |

### A.2.7 Findings

**1. Revenue leadership was fragmented in 2017 and converged in 2018.**
- In SP, the largest market, the #1 category holds only 9.4–12.1% of quarterly revenue, so no single category dominates. Leadership rotated through 2017: furniture_decor (Q1), bed_bath_table (Q2–Q3), then watches_gifts (Q4). bed_bath_table is SP's most consistent driver, in the top 3 in all 7 quarters.
- The all-state grid shows the same fragmentation nationally in 2017: in 2017 Q1 the 10 qualifying states had 6 different leading categories.
- In 2017 Q4, watches_gifts led in 6 of those 10 states, a national holiday-season effect.
- By 2018 leadership had converged. health_beauty or watches_gifts led in 17 of 21 states in 2018 Q2, and health_beauty alone led in 14 of 19 in 2018 Q3.
- Nationally, health_beauty is the clear leader: 84 top-3 appearances, ranked 1st 49 times, and in the top 3 of all 21 qualifying states. It is followed by watches_gifts (59 appearances, 27 firsts).
- The marketplace also expanded geographically. The number of states with at least one category reaching 10 orders grew from 10 in 2017 Q1 to all 21 in 2018 Q2. Newer, smaller markets (mainly North and Northeast) were led almost entirely by health_beauty or watches_gifts from the start.
- 2018 Q3 has lower absolute revenue and two states without a qualifying category, consistent with the dataset's coverage tailing off in that quarter (12,820 orders vs 19,979 in Q2). Revenue shares are therefore more reliable than absolute values for that quarter.re more reliable than absolute values for that quarter.

**2. Late delivery is strongly associated with lower review scores, in every segment.**
- Of 95,560 eligible delivered orders, 6.67% arrived after the estimated date.
- On-time orders average 4.29 stars and late orders 2.27, a gap of **2.02 points** on a 5-point scale.
- All 29 qualifying categories and all 21 qualifying states show a positive gap (1.63–2.62 in the states) and a negative correlation, so the effect is universal rather than confined to particular segments.
- The correlation between delay in days and review score is only −0.27, despite the large gap. Most orders arrive well *before* the estimate (mean delay −11.8 days), and within that early range extra days barely affect reviews. The relationship is a threshold effect (late vs not late) rather than a linear one, so the review gap is the more informative measure.

**3. Logistics problems are regional more than product-specific.**
- Late rates vary about fivefold across states: from 3.98% in PR and 4.44% in SP, up to 20.87% in AL, 17.21% in MA and 15.11% in SE. The five worst states (AL, MA, SE, PI, CE) are all in the Northeast, far from the South-East seller base.
- The correlation is also stronger where lateness is common (AL −0.47, CE −0.47, RJ −0.41) than where it is rare (SP −0.18, PR −0.18).
- Across categories, late rates vary far less (5.0–11.7% among the top 10 by review gap). Lateness costs the most review points in musical_instruments (2.51), audio (2.39, also the highest category late rate at 11.66%) and toys (2.24).

**4. Logistics vs demand: Rio de Janeiro stands out.**
- Of the 324 top-3 slots, 24 are flagged as a logistics concern, 42 show no concern, and 258 lack enough orders for a reliable state × category estimate. Most of those are in smaller states, where 30 late orders per category are rarely reached, which is a limitation of assessing at this granularity. This is partly structural: as the leadership grid shows, many smaller states only reach meaningful order volumes from mid-2017 onwards.
- **21 of the 24 flagged slots are in RJ**, across 8 categories: led by watches_gifts (top 3 in 6 quarters, 13.72% late), health_beauty (5 quarters) and bed_bath_table (3 quarters, 14.75% late). Together these flagged RJ slots represent about R$500k of top-3 revenue.
- RJ's late rate (11.96%) is almost double the baseline (6.67%), and its review gap (2.33) is above the baseline (2.02).
- Demand in RJ is clearly strong, since these are its top revenue earners. Satisfaction is being eroded by delivery performance, which makes RJ logistics the clearest target for operational investment.
- In contrast, SP and MG, the two other largest markets, have below-baseline late rates (4.44% and 4.49%) and no flagged slots.



### A.2.8 How Catalyst optimises this query
The points below describe how filtering, projection and join ordering shape the execution plan. They are verified against the `explain()` output in Part B.

**Filtering (predicate pushdown).**
- The date-range and status filter on `orders` is applied before any join, and Catalyst pushes it into the scan (`PushedFilters` on the `orders` FileScan). For CSV, rows are discarded while being parsed, so they never reach the join or the shuffle. The whole file is still read from disk, because CSV has no per-file statistics (unlike Parquet).
- The `n_orders >= MIN_GROUP_ORDERS` filter **cannot** be pushed down, because it depends on an aggregate result. It runs after the `HashAggregate`, which is exactly why it is a HAVING-style step.

**Projection (column pruning).**
- Each table is narrowed to the columns needed before joining: for example, 3 of 7 `order_items` columns and 2 of 5 `customers` columns. Catalyst would prune unused columns anyway (the scan's `ReadSchema` lists only required fields, and the CSV parser skips converting the rest).
- Explicit `select()` calls make this visible and keep rows narrow through every shuffle. Shuffle cost is driven by bytes moved, not just row count.

**Join strategy and ordering.**
- Cost-based join reordering is off by default (`spark.sql.cbo.enabled = false`), and no table statistics exist, so Catalyst keeps joins in the order written.
- The pipeline therefore does the largest join first, between `order_items` (112,650 rows) and the already-filtered `orders`. On two large inputs this is expected to be a `SortMergeJoin`, with both sides shuffled on `order_id`.
- The small dimensions are joined afterwards with **broadcast** hints (`products` with 2 columns kept, and `category_translation` at 71 rows). Each task probes an in-memory copy, so the large side is not shuffled again.
- With AQE enabled, Spark may also convert a sort-merge join into a broadcast join at runtime, if a side's actual shuffled size turns out to be under the 10 MB threshold. The initial and final plans can therefore differ.

**Caching.**
- `base_df` is reused by the threshold check, both revenue aggregations and four review aggregations. Caching it replaces a repeated six-CSV scan and five joins in each job's DAG with a single `InMemoryTableScan`.

## A.3 Spark SQL Implementation

In [29]:
# ---------------------------------------------------------------
# Register raw tables as temporary views
# ---------------------------------------------------------------
for name, df in tables.items():
    df.createOrReplaceTempView(name)

spark.sql("SHOW VIEWS").show(truncate=False)

+---------+--------------------+-----------+
|namespace|viewName            |isTemporary|
+---------+--------------------+-----------+
|         |category_translation|true       |
|         |customers           |true       |
|         |order_items         |true       |
|         |order_reviews       |true       |
|         |orders              |true       |
|         |products            |true       |
+---------+--------------------+-----------+



In [30]:
# ---------------------------------------------------------------
# Base table (SQL equivalent of base_df), cached
# ---------------------------------------------------------------
excluded_sql = ", ".join(f"'{s}'" for s in EXCLUDED_STATUSES)

spark.sql("DROP VIEW IF EXISTS sql_base")   # allows re-running this cell
spark.sql(f'''
CACHE TABLE sql_base AS
WITH
-- Step 1: one review score per order
reviews_per_order AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
),
-- Step 2: filter and project orders before joining
filtered_orders AS (
    SELECT order_id, customer_id, order_status, order_purchase_timestamp,
           order_delivered_customer_date, order_estimated_delivery_date
    FROM orders
    WHERE order_purchase_timestamp >= TIMESTAMP '{START_TS}'
      AND order_purchase_timestamp <  TIMESTAMP '{END_TS}'
      AND order_status NOT IN ({excluded_sql})
),
-- Step 3: joins and derived columns
enriched AS (
    SELECT /*+ BROADCAST(p), BROADCAST(t) */
        i.order_id,
        c.customer_state,
        format_string('%d-Q%d', year(o.order_purchase_timestamp),
                                quarter(o.order_purchase_timestamp))       AS year_quarter,
        COALESCE(t.product_category_name_english,
                 p.product_category_name, 'uncategorised')                 AS category,
        i.price,
        o.order_status,
        CASE WHEN o.order_status = 'delivered'
             THEN datediff(to_date(o.order_delivered_customer_date),
                           to_date(o.order_estimated_delivery_date))
        END                                                                 AS delay_days,
        r.review_score
    FROM order_items i
    JOIN filtered_orders o           ON i.order_id = o.order_id
    JOIN customers c                 ON o.customer_id = c.customer_id
    LEFT JOIN products p             ON i.product_id = p.product_id
    LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
    LEFT JOIN reviews_per_order r    ON i.order_id = r.order_id
)
-- Step 4: is_late depends on delay_days, so it is added in an outer SELECT
SELECT order_id, customer_state, year_quarter, category, price, order_status,
       delay_days, delay_days > 0 AS is_late, review_score
FROM enriched
''')

print(f"sql_base cached: {spark.table('sql_base').count():,} line items")

sql_base cached: 111,753 line items


In [31]:
# ---------------------------------------------------------------
# Analysis 1: top revenue categories per state and quarter
# ---------------------------------------------------------------
top_categories_sql = spark.sql(f'''
WITH
-- Revenue per (state, quarter, category), kept only above the volume threshold
category_groups AS (
    SELECT customer_state, year_quarter, category,
           ROUND(SUM(price), 2)     AS revenue,
           COUNT(DISTINCT order_id) AS n_orders,
           COUNT(*)                 AS n_items
    FROM sql_base
    GROUP BY customer_state, year_quarter, category
    HAVING COUNT(DISTINCT order_id) >= {MIN_GROUP_ORDERS}
),
-- Total revenue per (state, quarter), before the threshold
state_quarter_totals AS (
    SELECT customer_state, year_quarter, SUM(price) AS state_quarter_revenue
    FROM sql_base
    GROUP BY customer_state, year_quarter
),
-- Share of state-quarter revenue, and rank within each (state, quarter)
ranked AS (
    SELECT /*+ BROADCAST(t) */
        g.customer_state, g.year_quarter, g.category, g.revenue, g.n_orders, g.n_items,
        ROUND(100 * g.revenue / t.state_quarter_revenue, 2) AS revenue_share_pct,
        RANK() OVER (PARTITION BY g.customer_state, g.year_quarter
                     ORDER BY g.revenue DESC, g.category ASC) AS revenue_rank
    FROM category_groups g
    JOIN state_quarter_totals t
      ON g.customer_state = t.customer_state AND g.year_quarter = t.year_quarter
)
SELECT customer_state, year_quarter, revenue_rank, category,
       revenue, revenue_share_pct, n_orders, n_items
FROM ranked
WHERE revenue_rank <= {TOP_N}
''')
top_categories_sql.createOrReplaceTempView("sql_top_categories")

print(f"Top-{TOP_N} rows across all states and quarters: {top_categories_sql.count()}")
spark.sql('''
SELECT * FROM sql_top_categories
WHERE customer_state = 'SP'
ORDER BY year_quarter, revenue_rank
''').show(3 * 7, truncate=False)

Top-3 rows across all states and quarters: 324
+--------------+------------+------------+---------------------+---------+-----------------+--------+-------+
|customer_state|year_quarter|revenue_rank|category             |revenue  |revenue_share_pct|n_orders|n_items|
+--------------+------------+------------+---------------------+---------+-----------------+--------+-------+
|SP            |2017-Q1     |1           |furniture_decor      |23819.56 |9.38             |274     |340    |
|SP            |2017-Q1     |2           |sports_leisure       |23287.74 |9.17             |144     |175    |
|SP            |2017-Q1     |3           |bed_bath_table       |19370.73 |7.63             |189     |215    |
|SP            |2017-Q2     |1           |bed_bath_table       |40435.78 |8.46             |397     |453    |
|SP            |2017-Q2     |2           |computers_accessories|33516.35 |7.01             |218     |257    |
|SP            |2017-Q2     |3           |cool_stuff           |32498.33 

In [32]:
# #1 revenue category for every (state, quarter):
leaders_sql = spark.sql('''
WITH
-- The #1 category in each (state, quarter)
leaders AS (
    SELECT customer_state, year_quarter, category
    FROM sql_top_categories
    WHERE revenue_rank = 1
),
-- How many quarters each state has a leader in (used for ordering)
quarters_filled AS (
    SELECT customer_state, COUNT(*) AS n_quarters
    FROM leaders
    GROUP BY customer_state
),
-- One row per state, one column per quarter
pivoted AS (
    SELECT * FROM leaders
    PIVOT (
        FIRST(category) FOR year_quarter IN (
            '2017-Q1' AS `2017-Q1`, '2017-Q2' AS `2017-Q2`,
            '2017-Q3' AS `2017-Q3`, '2017-Q4' AS `2017-Q4`,
            '2018-Q1' AS `2018-Q1`, '2018-Q2' AS `2018-Q2`,
            '2018-Q3' AS `2018-Q3`
        )
    )
)
-- '-' = no category met the minimum-orders threshold
SELECT p.customer_state,
       COALESCE(p.`2017-Q1`, '-') AS `2017-Q1`,
       COALESCE(p.`2017-Q2`, '-') AS `2017-Q2`,
       COALESCE(p.`2017-Q3`, '-') AS `2017-Q3`,
       COALESCE(p.`2017-Q4`, '-') AS `2017-Q4`,
       COALESCE(p.`2018-Q1`, '-') AS `2018-Q1`,
       COALESCE(p.`2018-Q2`, '-') AS `2018-Q2`,
       COALESCE(p.`2018-Q3`, '-') AS `2018-Q3`
FROM pivoted p
JOIN quarters_filled q ON p.customer_state = q.customer_state
ORDER BY q.n_quarters DESC, p.customer_state
''')

# Display as a table, matching the DataFrame version
leaders_sql.toPandas().set_index("customer_state")

,2017-Q1,2017-Q2,2017-Q3,2017-Q4,2018-Q1,2018-Q2,2018-Q3
customer_state,,,,,,,
BA,health_beauty,computers_accessories,health_beauty,watches_gifts,sports_leisure,watches_gifts,health_beauty
DF,sports_leisure,sports_leisure,health_beauty,watches_gifts,watches_gifts,health_beauty,watches_gifts
ES,sports_leisure,health_beauty,bed_bath_table,watches_gifts,computers_accessories,watches_gifts,watches_gifts
GO,cool_stuff,cool_stuff,bed_bath_table,watches_gifts,health_beauty,watches_gifts,health_beauty
MG,furniture_decor,cool_stuff,bed_bath_table,watches_gifts,health_beauty,health_beauty,health_beauty
PR,cool_stuff,sports_leisure,sports_leisure,furniture_decor,computers_accessories,watches_gifts,watches_gifts
RJ,furniture_decor,health_beauty,bed_bath_table,bed_bath_table,watches_gifts,watches_gifts,health_beauty
RS,health_beauty,bed_bath_table,bed_bath_table,computers_accessories,sports_leisure,bed_bath_table,health_beauty
SC,garden_tools,computers_accessories,health_beauty,sports_leisure,sports_leisure,furniture_decor,health_beauty


In [33]:
# ---------------------------------------------------------------
# Analysis 2: delivery delay vs review score (two-stage aggregation)
# ---------------------------------------------------------------
def logistics_sql(dims):
    # Build the SQL for one segmentation level; dims = [] gives the overall baseline.
    dim_cols   = "".join(f"{d}, " for d in dims)
    group_by   = f"GROUP BY {', '.join(dims)}" if dims else ""
    return f'''
    WITH
    -- Stage 1: one row per order within each segment
    order_level AS (
        SELECT DISTINCT order_id, {dim_cols}delay_days, is_late, review_score
        FROM sql_base
        WHERE delay_days IS NOT NULL AND review_score IS NOT NULL
    )
    -- Stage 2: aggregate per segment, keep segments with enough evidence
    SELECT {dim_cols}
        COUNT(*)                                              AS n_orders,
        SUM(CAST(is_late AS INT))                             AS n_late,
        ROUND(100 * AVG(CAST(is_late AS INT)), 2)             AS late_rate_pct,
        ROUND(AVG(delay_days), 2)                             AS avg_delay_days,
        ROUND(AVG(CASE WHEN NOT is_late THEN review_score END), 3) AS avg_review_on_time,
        ROUND(AVG(CASE WHEN is_late     THEN review_score END), 3) AS avg_review_late,
        ROUND(AVG(CASE WHEN NOT is_late THEN review_score END)
            - AVG(CASE WHEN is_late     THEN review_score END), 3) AS review_gap,
                -- Pearson correlation; try_divide returns NULL for zero-spread groups
        ROUND(try_divide(COVAR_SAMP(CAST(delay_days AS DOUBLE), review_score),
                         STDDEV_SAMP(CAST(delay_days AS DOUBLE)) * STDDEV_SAMP(review_score)), 3)
                                                              AS corr_delay_review
    FROM order_level
    {group_by}
    HAVING COUNT(*) >= {MIN_SEGMENT_ORDERS}
       AND SUM(CAST(is_late AS INT)) >= {MIN_LATE_ORDERS}
    '''

overall_logistics_sql      = spark.sql(logistics_sql([]))
logistics_by_category_sql  = spark.sql(logistics_sql(["category"]))
logistics_by_state_sql     = spark.sql(logistics_sql(["customer_state"]))
logistics_by_state_cat_sql = spark.sql(logistics_sql(["customer_state", "category"]))

for view, df in [("sql_overall_logistics", overall_logistics_sql),
                 ("sql_logistics_by_category", logistics_by_category_sql),
                 ("sql_logistics_by_state", logistics_by_state_sql),
                 ("sql_logistics_by_state_cat", logistics_by_state_cat_sql)]:
    df.createOrReplaceTempView(view)

print("Generated SQL for the category level:")
print(logistics_sql(["category"]))

print("Overall baseline:")
spark.sql("SELECT * FROM sql_overall_logistics").show(truncate=False)

print("Categories where lateness costs the most review points (top 10):")
spark.sql('''
SELECT * FROM sql_logistics_by_category
ORDER BY review_gap DESC, category ASC
LIMIT 10
''').show(truncate=False)

Generated SQL for the category level:

    WITH
    -- Stage 1: one row per order within each segment
    order_level AS (
        SELECT DISTINCT order_id, category, delay_days, is_late, review_score
        FROM sql_base
        WHERE delay_days IS NOT NULL AND review_score IS NOT NULL
    )
    -- Stage 2: aggregate per segment, keep segments with enough evidence
    SELECT category, 
        COUNT(*)                                              AS n_orders,
        SUM(CAST(is_late AS INT))                             AS n_late,
        ROUND(100 * AVG(CAST(is_late AS INT)), 2)             AS late_rate_pct,
        ROUND(AVG(delay_days), 2)                             AS avg_delay_days,
        ROUND(AVG(CASE WHEN NOT is_late THEN review_score END), 3) AS avg_review_on_time,
        ROUND(AVG(CASE WHEN is_late     THEN review_score END), 3) AS avg_review_late,
        ROUND(AVG(CASE WHEN NOT is_late THEN review_score END)
            - AVG(CASE WHEN is_late     THEN review_score EN

In [34]:
# ---------------------------------------------------------------
# Analysis 3: logistics issue or demand issue?
# ---------------------------------------------------------------
top_with_logistics_sql = spark.sql('''
WITH
baseline AS (
    SELECT late_rate_pct AS baseline_late_rate_pct,
           review_gap    AS baseline_review_gap
    FROM sql_overall_logistics
),
joined AS (
    SELECT /*+ BROADCAST(b) */
        t.*, l.n_orders AS logistics_orders, l.late_rate_pct, l.review_gap,
        b.baseline_late_rate_pct, b.baseline_review_gap
    FROM sql_top_categories t
    LEFT JOIN sql_logistics_by_state_cat l
      ON t.customer_state = l.customer_state AND t.category = l.category
    CROSS JOIN baseline b
)
SELECT customer_state, year_quarter, revenue_rank, category, revenue,
       revenue_share_pct, logistics_orders, late_rate_pct, review_gap,
       CASE WHEN late_rate_pct IS NULL THEN 'insufficient data'
            WHEN late_rate_pct > baseline_late_rate_pct
             AND review_gap    > baseline_review_gap THEN 'logistics concern'
            ELSE 'no concern'
       END AS logistics_flag
FROM joined
''')
top_with_logistics_sql.createOrReplaceTempView("sql_top_with_logistics")

spark.sql('''
SELECT logistics_flag, COUNT(*) AS count
FROM sql_top_with_logistics
GROUP BY logistics_flag
ORDER BY count DESC
''').show()

+-----------------+-----+
|   logistics_flag|count|
+-----------------+-----+
|insufficient data|  258|
|       no concern|   42|
|logistics concern|   24|
+-----------------+-----+



In [35]:
print("Top-revenue category-state combinations flagged as a logistics concern:")
spark.sql('''
SELECT customer_state,
       category,
       COUNT(*)                 AS quarters_in_top3,
       ROUND(SUM(revenue), 2)   AS top3_revenue,
       FIRST(late_rate_pct)     AS late_rate_pct,
       FIRST(review_gap)        AS review_gap
FROM sql_top_with_logistics
WHERE logistics_flag = 'logistics concern'
GROUP BY customer_state, category
ORDER BY top3_revenue DESC
LIMIT 15
''').show(truncate=False)

Top-revenue category-state combinations flagged as a logistics concern:
+--------------+---------------------+----------------+------------+-------------+----------+
|customer_state|category             |quarters_in_top3|top3_revenue|late_rate_pct|review_gap|
+--------------+---------------------+----------------+------------+-------------+----------+
|RJ            |watches_gifts        |6               |162832.27   |13.72        |2.091     |
|RJ            |health_beauty        |5               |97623.47    |8.74         |2.327     |
|RJ            |bed_bath_table       |3               |87221.28    |14.75        |2.12      |
|RJ            |computers_accessories|2               |55517.77    |11.89        |2.498     |
|RJ            |sports_leisure       |2               |43943.67    |13.84        |2.395     |
|RS            |bed_bath_table       |3               |33087.57    |8.43         |2.061     |
|RJ            |toys                 |1               |29546.79    |10.61        |

## A.4 Result validation

Both implementations are compared systematically for every result they produce, not by eye.

| Check | Method |
|---|---|
| Same columns and types | Compare `dtypes` |
| Same number of rows | `count()` on both |
| Same rows (as a multiset) | `exceptAll()` in both directions. `exceptAll` keeps duplicates, unlike `subtract`, so a row appearing twice on one side and once on the other is still caught. |
| Numeric agreement | Join on the key columns and report the largest absolute difference in each numeric column |
| Leader grid | The pandas pivot (DataFrame API) and the `PIVOT` query (Spark SQL) are compared cell by cell with `DataFrame.equals` |

The seven Spark outputs validated here are the sources of every displayed table. The SP view, the national summary and the flagged-combination breakdown are simple filters and groupings of these outputs, so they are equivalent whenever their sources are.

`exceptAll` compares rows regardless of their order. This matters because row order is **not** part of either result. Without a final `orderBy` / `ORDER BY`, rows come back in whatever order the tasks of the last stage produce them.

In [36]:
# ---------------------------------------------------------------
# Systematic comparison of every DataFrame-API output with its
# Spark SQL equivalent
# ---------------------------------------------------------------
def validate(name, df_a, df_b, keys):
    """Compare a DataFrame-API result (df_a) with its Spark SQL equivalent (df_b)."""
    numeric = [c for c, t in df_a.dtypes
               if t in ("double", "float", "int", "bigint") and c not in keys]

    # Order-insensitive row comparison, in both directions
    a_minus_b = df_a.exceptAll(df_b).count()
    b_minus_a = df_b.exceptAll(df_a).count()

    # Largest absolute difference per numeric column, after aligning rows on the keys.
    # Outputs without a key are either one row (cross join is safe) or the base
    # table, where the exact exceptAll comparison above is already sufficient.
    n_a, n_b = df_a.count(), df_b.count()
    if keys:
        aligned = df_a.alias("a").join(df_b.alias("b"), keys, "full_outer")
    elif n_a == n_b == 1:
        aligned = df_a.alias("a").crossJoin(df_b.alias("b"))
    else:
        aligned = None
    if aligned is not None and numeric:
        diffs = aligned.select([F.max(F.abs(F.col(f"a.{c}") - F.col(f"b.{c}"))).alias(c)
                                for c in numeric]).first()
        max_diff = max((v for v in diffs if v is not None), default=0.0)
    else:
        max_diff = "n/a (exact match checked)"

    return {
        "Output": name,
        "Same schema": df_a.dtypes == df_b.dtypes,
        "Rows (DataFrame)": n_a,
        "Rows (SQL)": n_b,
        "Only in DataFrame": a_minus_b,
        "Only in SQL": b_minus_a,
        "Max numeric diff": max_diff,
        "Equivalent": df_a.dtypes == df_b.dtypes and a_minus_b == 0 and b_minus_a == 0,
    }


results = [
    validate("base table", base_df, spark.table("sql_base"), []),
    validate("top_categories", top_categories_df, top_categories_sql,
             ["customer_state", "year_quarter", "category"]),
    validate("overall_logistics", overall_logistics_df, overall_logistics_sql, []),
    validate("logistics_by_category", logistics_by_category_df, logistics_by_category_sql,
             ["category"]),
    validate("logistics_by_state", logistics_by_state_df, logistics_by_state_sql,
             ["customer_state"]),
    validate("logistics_by_state_cat", logistics_by_state_cat_df, logistics_by_state_cat_sql,
             ["customer_state", "category"]),
    validate("top_with_logistics", top_with_logistics_df, top_with_logistics_sql,
             ["customer_state", "year_quarter", "category"]),
]
pd.DataFrame(results)

,Output,Same schema,Rows (DataFrame),Rows (SQL),Only in DataFrame,Only in SQL,Max numeric diff,Equivalent
0,base table,True,111753,111753,0,0,n/a (exact match checked),True
1,top_categories,True,324,324,0,0,0,True
2,overall_logistics,True,1,1,0,0,0,True
3,logistics_by_category,True,29,29,0,0,0,True
4,logistics_by_state,True,21,21,0,0,0,True
5,logistics_by_state_cat,True,42,42,0,0,0,True
6,top_with_logistics,True,324,324,0,0,0,True


In [37]:
# ---------------------------------------------------------------
# Leader grid: pandas pivot (DataFrame API) vs SQL PIVOT
# ---------------------------------------------------------------
# Both grids are small pandas tables, so they are compared cell by cell.
leaders_sql_pd = leaders_sql.toPandas().set_index("customer_state")
leaders_sql_pd.columns.name = leaders_pd.columns.name   # align the header label only

print("Same shape:        ", leaders_pd.shape, "vs", leaders_sql_pd.shape)
print("Same state order:  ", list(leaders_pd.index) == list(leaders_sql_pd.index))
print("Identical grids:   ", leaders_pd.equals(leaders_sql_pd))

Same shape:         (21, 7) vs (21, 7)
Same state order:   True
Identical grids:    True


In [38]:
# Row order is not part of either result. Without orderBy / ORDER BY the
# order depends on partitioning and task scheduling, not on the query text.
print("DataFrame API, no orderBy:")
top_categories_df.show(5, truncate=False)
print("Spark SQL, no ORDER BY:")
top_categories_sql.show(5, truncate=False)

DataFrame API, no orderBy:
+--------------+------------+------------+--------------+-------+-----------------+--------+-------+
|customer_state|year_quarter|revenue_rank|category      |revenue|revenue_share_pct|n_orders|n_items|
+--------------+------------+------------+--------------+-------+-----------------+--------+-------+
|AL            |2017-Q2     |1           |health_beauty |1332.13|12.76            |11      |11     |
|AL            |2017-Q3     |1           |health_beauty |3936.23|25.11            |10      |10     |
|AL            |2017-Q4     |1           |health_beauty |1723.11|12.76            |10      |10     |
|AL            |2018-Q1     |1           |health_beauty |2044.95|12.23            |13      |13     |
|AL            |2018-Q1     |2           |sports_leisure|1130.79|6.76             |10      |11     |
+--------------+------------+------------+--------------+-------+-----------------+--------+-------+
only showing top 5 rows
Spark SQL, no ORDER BY:
+--------------+

In [41]:
# ---------------------------------------------------------------
# Correctness checks: do the results reconcile with the raw data?
# ---------------------------------------------------------------
# Reference: order items belonging to orders that pass the date/status filter,
# computed directly from the raw tables without any of the enrichment joins.
raw_items = order_items_df.join(filtered_orders_df.select("order_id"), "order_id")
raw_rows, raw_revenue = raw_items.agg(F.count("*"), F.sum("price")).first()

base_rows, base_revenue = base_df.agg(F.count("*"), F.sum("price")).first()
sq_total = state_quarter_totals_df.agg(F.sum("state_quarter_revenue")).first()[0]
eligible_orders = review_base_df.select("order_id").distinct().count()
top_per_pair = top_categories_df.groupBy("customer_state", "year_quarter").count()

checks = [
    ("Base rows = filtered raw order items (no rows lost or duplicated by joins)",
     raw_rows, base_rows, raw_rows == base_rows),
    ("Base revenue = filtered raw revenue",
     round(raw_revenue, 2), round(base_revenue, 2), abs(raw_revenue - base_revenue) < 0.01),
    ("State-quarter totals sum to base revenue",
     round(base_revenue, 2), round(sq_total, 2), abs(base_revenue - sq_total) < 0.01),
    ("Orders with more than one review after pre-aggregation",
     0, reviews_per_order_df.groupBy("order_id").count().filter("count > 1").count(), None),
    ("Overall n_orders = distinct eligible orders (stage-1 dedup worked)",
     eligible_orders, overall_logistics_df.first()["n_orders"], None),
    (f"(state, quarter) pairs with more than {TOP_N} top rows",
     0, top_per_pair.filter(F.col("count") > TOP_N).count(), None),
    (f"Top-{TOP_N} rows below MIN_GROUP_ORDERS",
     0, top_categories_df.filter(F.col("n_orders") < MIN_GROUP_ORDERS).count(), None),
]
checks = [(c, e, a, (e == a) if ok is None else ok) for c, e, a, ok in checks]
pd.DataFrame(checks, columns=["Check", "Expected", "Actual", "Pass"])

,Check,Expected,Actual,Pass
0,Base rows = filtered raw order items (no rows ...,111753.00,111753.00,True
1,Base revenue = filtered raw revenue,13449674.68,13449674.68,True
2,State-quarter totals sum to base revenue,13449674.68,13449674.68,True
3,Orders with more than one review after pre-agg...,0.00,0.00,True
4,Overall n_orders = distinct eligible orders (s...,95560.00,95560.00,True
5,"(state, quarter) pairs with more than 3 top rows",0.00,0.00,True
6,Top-3 rows below MIN_GROUP_ORDERS,0.00,0.00,True


In [40]:
# ---------------------------------------------------------------
# Partition invariance: results must not depend on how data is split
# ---------------------------------------------------------------
# spark.sql.shuffle.partitions is read when a query is planned, so each run
# builds a fresh plan (select("*")) and compares it with the SQL result
# that was computed with the original 16 partitions.
original_partitions = spark.conf.get("spark.sql.shuffle.partitions")
rows = []
try:
    for n in [1, 4, 64]:
        spark.conf.set("spark.sql.shuffle.partitions", n)
        for name, df_a, df_b in [
            ("top_categories", top_categories_df, top_categories_sql),
            ("logistics_by_state_cat", logistics_by_state_cat_df, logistics_by_state_cat_sql),
        ]:
            fresh = df_a.select("*")
            rows.append({
                "Shuffle partitions": n,
                "Output": name,
                "Only in re-run": fresh.exceptAll(df_b).count(),
                "Only in original": df_b.exceptAll(fresh).count(),
            })
finally:
    spark.conf.set("spark.sql.shuffle.partitions", original_partitions)  # always restore

print(f"Shuffle partitions restored to {spark.conf.get('spark.sql.shuffle.partitions')}")
pd.DataFrame(rows)

Shuffle partitions restored to 16


,Shuffle partitions,Output,Only in re-run,Only in original
0,1,top_categories,0,0
1,1,logistics_by_state_cat,0,0
2,4,top_categories,0,0
3,4,logistics_by_state_cat,0,0
4,64,top_categories,0,0
5,64,logistics_by_state_cat,0,0


### Validation results

**Equivalence.** All seven outputs are identical between the two APIs: same schema, same row counts, no rows unique to either side, and a maximum numeric difference of 0. The leader grids also match cell by cell. This is expected, because both APIs are optimised by Catalyst into the same plan.

**Correctness.** The results also reconcile with the raw data:
- The base table has exactly one row per filtered order item (111,753), so no rows were lost or duplicated by the joins.
- Revenue matches to the cent at every level (R$13,449,674.68).
- Each order is counted once in the review analysis (95,560 orders).
- No (state, quarter) pair has more than 3 rows, and no row falls below the minimum-orders threshold.

**Partition invariance.** Results are identical with 1, 4 and 64 shuffle partitions. Partitioning changes how the work is split, not the answer.

**Why minor differences did not occur:**
- **Sorting.** Without `orderBy`, row order depends on partitioning and task scheduling, so it is not guaranteed. Both APIs happened to return the same order because they ran the same plan. The validation uses order-insensitive `exceptAll` for this reason.
- **Floating point.** Sums of `double` values can differ in the last digits depending on the order in which partitions are combined. Both versions round at the same points, which removes this noise.
- **Ranking ties.** Ranking on rounded revenue, with `category` as a tie-breaker, keeps ranks identical even when two categories have equal revenue.
- **Correlation.** Spark 4's ANSI mode raised divide-by-zero errors for `corr()` on groups with no variation, so both versions compute it the same way using `try_divide`.
- **Formatting.** The pandas grid carries a `year_quarter` column label that the SQL grid lacks, and `show()` drops trailing zeros (`77803.7`). These are display differences only, not differences in value.

## A.5 DataFrame API vs Spark SQL

**Structure.** Both follow the same steps. SQL names each step as a CTE (`filtered_orders`, `category_groups`, `ranked`); the DataFrame API uses intermediate variables (`filtered_orders_df`, `state_quarter_totals_df`) and method chains.

**Readability.**
- SQL is clearer for aggregation logic: `HAVING` and `RANK() OVER (...)` state their purpose where they are used. In the DataFrame version, the HAVING step is a plain `.filter()`, and the window is defined separately from where it is used.
- The DataFrame API is clearer for derived columns: `is_late` can use `delay_days` directly, whereas SQL needed an extra outer query.

**Maintainability.**
- The DataFrame API handles reuse better. `logistics_metrics(dims)` covers all four levels of Analysis 2 in one function, whereas SQL needed a function that generates query text.
- Parameters are native Python values in the DataFrame API, but must be formatted into SQL strings.
- The SQL `PIVOT` requires the quarters to be listed by hand, so it would need updating if the date range changed.

**Debuggability.** Any intermediate DataFrame can be checked with `.show()`. A CTE cannot be run on its own.

**Performance.** Neither is faster: both compile to the same plan, as the identical results and row order show.

**Conclusion.** SQL suits readable, step-by-step reporting logic like the revenue ranking. The DataFrame API suits reusable, parameterised logic like the logistics analysis. Since performance is the same, the choice comes down to readability and maintenance.